In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

df = pd.read_csv('../datasets/master/master_final_2025.csv')

In [2]:
# Features for clustering: access + outcome + gender gap + library (fill NaN with column mean for the 4 Papua provinces)
feat_cols = ['facility_ratio_jhs','facility_ratio_shs','school_density_primary',
            'library_per_1000pupils','literacy_total','eys_avg','gap_literacy','gap_eys']
X_raw = df[feat_cols].copy()
X_raw = X_raw.fillna(X_raw.mean())
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)


In [3]:
# --- Elbow + Silhouette to pick k ---
inertias, sils = [], []
K_range = range(2,8)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X, km.labels_))

fig, axes = plt.subplots(1,2, figsize=(12,4.5))
axes[0].plot(list(K_range), inertias, 'o-', color='#1565C0')
axes[0].set_xlabel('Jumlah cluster (k)'); axes[0].set_ylabel('Inertia'); axes[0].set_title('Elbow Method')
axes[1].plot(list(K_range), sils, 'o-', color='#E65100')
axes[1].set_xlabel('Jumlah cluster (k)'); axes[1].set_ylabel('Silhouette Score'); axes[1].set_title('Silhouette Score')
plt.suptitle('Pemilihan Jumlah Cluster Optimal', fontweight='bold')
plt.tight_layout()
plt.savefig('../src/visualization/06_elbow_silhouette.png', dpi=150)
plt.close()

In [4]:
print("Silhouette scores per k:", dict(zip(K_range, [round(s,3) for s in sils])))
best_k = list(K_range)[np.argmax(sils)]
print(f"Best k by silhouette: {best_k}")

Silhouette scores per k: {2: 0.482, 3: 0.367, 4: 0.276, 5: 0.291, 6: 0.166, 7: 0.133}
Best k by silhouette: 2


In [5]:
# Use k=4 for interpretability (matches the 4-quadrant framing the user wants), report silhouette for k=4 too
k_final = 4
km_final = KMeans(n_clusters=k_final, random_state=42, n_init=10).fit(X)
df['kmeans_cluster'] = km_final.labels_
print(f"\nSilhouette @k=4: {silhouette_score(X, km_final.labels_):.3f}")



Silhouette @k=4: 0.276


In [6]:
# PCA for visualization
pca = PCA(n_components=2)
pcs = pca.fit_transform(X)
df['pc1'], df['pc2'] = pcs[:,0], pcs[:,1]
print("PCA explained variance ratio:", pca.explained_variance_ratio_.round(3))

PCA explained variance ratio: [0.359 0.249]


In [7]:
# Cluster profile (mean of raw, unstandardized values)
profile = df.groupby('kmeans_cluster')[feat_cols].mean().round(2)
profile['n_provinsi'] = df.groupby('kmeans_cluster').size()
print("\n=== Cluster profile (rata-rata nilai asli) ===")
print(profile)

print("\n=== Anggota tiap cluster ===")
for c in sorted(df['kmeans_cluster'].unique()):
    provs = df[df['kmeans_cluster']==c]['province'].str.title().tolist()
    print(f"Cluster {c} (n={len(provs)}): {provs}")

df.to_csv('../datasets/master/master_final_2025.csv', index=False)


=== Cluster profile (rata-rata nilai asli) ===
                facility_ratio_jhs  facility_ratio_shs  \
kmeans_cluster                                           
0                             0.62                0.31   
1                             0.76                0.47   
2                             0.48                0.21   
3                             0.32                0.10   

                school_density_primary  library_per_1000pupils  \
kmeans_cluster                                                   
0                                 6.97                    4.77   
1                                 4.08                    0.82   
2                                 7.66                    1.13   
3                                 4.28                    0.00   

                literacy_total  eys_avg  gap_literacy  gap_eys  n_provinsi  
kmeans_cluster                                                              
0                        95.35    13.78         -3.7